In [49]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import pandas as pd
from google import genai
import openai


In [50]:
df = pd.read_json('../data/Data_With_Details.jsonl', lines=True)
print(df.head())

                              review_id product_id  \
0  49625377-e444-4d3e-8e33-780aaa3ee59b       B002   
1  583c2647-a5cb-4d56-a259-8f6a88a836f8       B001   
2  beb8027a-f4a7-45d7-b942-82fe92875307       B005   
3  e9218ec2-a71e-4730-829b-a845d98002dc       B002   
4  e286680f-5b2a-4c61-8d43-610405182a29       B010   

                   product_name reviewer_id  rating     review_title  \
0  Stainless Steel Water Bottle     R719176       1  Could be better   
1              Wireless Earbuds     R331148       1  Could be better   
2             Bluetooth Speaker     R207175       1    Pretty decent   
3  Stainless Steel Water Bottle     R182627       3  Amazing quality   
4           Mechanical Keyboard     R183667       3  Could be better   

                                         review_text  verified_purchase  \
0  This product has highly recommended. could be ...              False   
1     This product has easy to use. could be better.              False   
2  This product h

In [51]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1584 entries, 0 to 1583
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   review_id          1584 non-null   str           
 1   product_id         1584 non-null   str           
 2   product_name       1584 non-null   str           
 3   reviewer_id        1584 non-null   str           
 4   rating             1584 non-null   int64         
 5   review_title       1584 non-null   str           
 6   review_text        1584 non-null   str           
 7   verified_purchase  1584 non-null   bool          
 8   helpful_votes      1584 non-null   int64         
 9   review_date        1584 non-null   str           
 10  date               1584 non-null   datetime64[us]
 11  details            1584 non-null   object        
dtypes: bool(1), datetime64[us](1), int64(2), object(1), str(7)
memory usage: 137.8+ KB


In [52]:
def preprocess_review(review):
    return {
        'review_id': review['review_id'],
        'product_id': review['product_id'],
        'review_text': review['review_text'],
        'rating': review['rating']
    }

In [53]:
def extract_large_review_text(review):
    review_text = review.get('review_text')

    # Keep plain text as-is.
    if isinstance(review_text, str):
        return review_text

    # If review_text is a dict, prefer the 'large' field and otherwise
    # return the longest string value we can find.
    if isinstance(review_text, dict):
        large_text = review_text.get('large')
        if isinstance(large_text, str) and large_text:
            return large_text

        string_values = [value for value in review_text.values() if isinstance(value, str)]
        if string_values:
            return max(string_values, key=len)
        return str(review_text)

    # If review_text is a list, return the longest string entry.
    if isinstance(review_text, list):
        string_values = [value for value in review_text if isinstance(value, str)]
        if string_values:
            return max(string_values, key=len)
        return ' '.join(map(str, review_text))

    return '' if review_text is None else str(review_text)


In [54]:
df.head()

,review_id,product_id,product_name,reviewer_id,rating,review_title,review_text,verified_purchase,helpful_votes,review_date,date,details
0,49625377-e444-4d3e-8e33-780aaa3ee59b,B002,Stainless Steel Water Bottle,R719176,1,Could be better,This product has highly recommended. could be ...,False,2,2024-04-24,2024-04-24,"{'Date First Available': 'April 24, 2024', 'Ma..."
1,583c2647-a5cb-4d56-a259-8f6a88a836f8,B001,Wireless Earbuds,R331148,1,Could be better,This product has easy to use. could be better.,False,37,2024-09-15,2024-09-15,"{'Date First Available': 'September 15, 2024',..."
2,beb8027a-f4a7-45d7-b942-82fe92875307,B005,Bluetooth Speaker,R207175,1,Pretty decent,This product has good value for money. pretty ...,True,24,2024-06-23,2024-06-23,"{'Date First Available': 'June 23, 2024', 'Man..."
3,e9218ec2-a71e-4730-829b-a845d98002dc,B002,Stainless Steel Water Bottle,R182627,3,Amazing quality,This product has battery life could be better....,False,40,2024-01-23,2024-01-23,"{'Date First Available': 'January 23, 2024', '..."
4,e286680f-5b2a-4c61-8d43-610405182a29,B010,Mechanical Keyboard,R183667,3,Could be better,This product has customer support was helpful....,True,6,2024-12-26,2024-12-26,"{'Date First Available': 'December 26, 2024', ..."


In [55]:
data_to_embed = df[['product_name', 'review_text', 'rating', 'details']].sample(n=100, random_state=42)

Define the embedding function

In [56]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')

# Verify keys are loaded
print(f"OpenAI API Key present: {bool(openai_api_key)}")
print(f"Google API Key present: {bool(google_api_key)}")
print(f"Qdrant URL present: {bool(qdrant_url)}")
print(f"Qdrant API Key present: {bool(qdrant_api_key)}")

OpenAI API Key present: True
Google API Key present: True
Qdrant URL present: False
Qdrant API Key present: False


In [57]:
from google import genai

client = genai.Client()

# Your review texts
review_texts = df['review_text'].fillna('').astype(str).tolist()

# Use an embedding model that is available in your environment.
embedding_model = 'models/gemini-embedding-2'

# Batch embedding function
def embed_batch(text_batch):
    response = client.models.embed_content(
        model=embedding_model,
        contents=text_batch,
    )
    return [embedding.values for embedding in response.embeddings]

# Process in batches
batch_size = 100
all_embeddings = []

for start in range(0, len(review_texts), batch_size):
    batch = review_texts[start:start + batch_size]
    try:
        embeddings = embed_batch(batch)
        all_embeddings.extend(embeddings)
        print(f"Processed {start + len(batch)} / {len(review_texts)} texts")
    except Exception as e:
        print(f"Error processing batch: {e}")
        break

# Store embeddings
if all_embeddings:
    df['embedding'] = all_embeddings
    print(f"Embedding dimension: {len(all_embeddings[0])}")
else:
    print("No embeddings generated")

Processed 100 / 1584 texts
Processed 200 / 1584 texts
Processed 300 / 1584 texts
Processed 400 / 1584 texts
Processed 500 / 1584 texts
Processed 600 / 1584 texts
Processed 700 / 1584 texts
Processed 800 / 1584 texts
Processed 900 / 1584 texts
Processed 1000 / 1584 texts
Processed 1100 / 1584 texts
Processed 1200 / 1584 texts
Processed 1300 / 1584 texts
Processed 1400 / 1584 texts
Processed 1500 / 1584 texts
Processed 1584 / 1584 texts


ValueError: Length of values (16) does not match length of index (1584)

In [ ]:
response = openai.embeddings.create(
    input = "Random Text"
    model = "text-embedding-3-small"
)

In [ ]:
len(response.data[0].embedding)

In [ ]:
def get_embedding(text):
    response = openai.embeddings.create(
        input = text,
        model = "text-embedding-3-small"
    )
    return response.data[0].embedding

In [ ]:
get_embedding("This is a sample review text to test the embedding function.")